# Day 8 - LLM foundations & API integration practical tasks
***

These tasks cover calling OpenAI API's and prompting.

## Explore how different model parameters and different prompts influence the response.
#### Here are some guidelines and advice you should consider:
* You can add as many cells as you need for each task.
* Use Markdown cells for your conclusions.
* For tasks that do not specify a model, you may choose any of the two available models: `gpt-5.6-luna` / `gpt-5.4-mini`.
* Within each comparison task, change only the model, prompt, or parameter specified in the instructions. Keep everything else - including the model, prompt, and other settings - the same across calls so you can compare the results fairly. For example, when testing different `temperature` values, use the same model, prompt, and output token limit (`max_output_tokens` for the Responses API or `max_completion_tokens` for the Chat Completions API).
* You can use both - [Responses API and Chat Completions API](https://medium.com/@sanathgoutham/openai-responses-api-vs-chat-completions-what-actually-changed-eeee86aff216) - to complete these tasks, but some model parameters are not supported by Responses API.
* LLMs understand [Markdown](https://www.markdownguide.org/cheat-sheet/) formatting in prompts. For example, wrapping text in double asterisks `(**important text**)` marks it as **bold** and can help signal emphasis.
* To copy text from a Markdown cell, open it in edit mode by clicking the pencil icon or double-clicking the cell.
* Optionally, paste API responses into a Markdown cell for better readability.
* Be creative and experiment.

In [13]:
import os
from dotenv import load_dotenv
from openai import OpenAI

if not os.getenv("OPENAI_API_KEY"):
    os.environ.pop("OPENAI_API_KEY", None)
load_dotenv()

True

In [14]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise RuntimeError("Set OPENAI_API_KEY in the W2D3 .env file before running this cell.")

client = OpenAI(base_url="https://ds-ai-internship.openai.azure.com/openai/v1/",
                api_key=OPENAI_API_KEY)

In [15]:
import time

def chat(prompt, model="gpt-5.4-mini", system=None, **kwargs):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    start = time.time()
    response = client.chat.completions.create(model=model, messages=messages, **kwargs)
    return response, time.time() - start

def show(response, seconds=None):
    for i, choice in enumerate(response.choices, 1):
        print(f"--- Response {i} ---\n{choice.message.content}\n")
    u = response.usage
    print(f"Input tokens: {u.prompt_tokens} | Output tokens: {u.completion_tokens}"
          + (f" | Time: {seconds:.1f}s" if seconds else ""))

In [16]:
import os
print("Proxy settings:", {k: v for k, v in os.environ.items() if "proxy" in k.lower()})

try:
    client.models.list()
    print("Connection OK")
except Exception as e:
    print("Error:", repr(e))
    cause = e.__cause__
    while cause:
        print("Caused by:", repr(cause))
        cause = cause.__cause__

Proxy settings: {}
Connection OK


Helper. chat() sends promt and prints show() answers plus token counts. Since each tasks uses these two, do it once and not 26 times. 

#### 1. Write a Python script to call the OpenAI API and generate a response to the prompt below.

Prompt: `"What is artificial intelligence?"`

Use model `gpt-5.6-luna`.

In [17]:
response = client.responses.create(model="gpt-5.6-luna", input="What is artificial intelligence?")
#print(response.output_text)

Artificial intelligence (AI) is the field of creating computer systems that can perform tasks that typically require human intelligence, such as:

- Understanding language
- Recognizing images or speech
- Learning from data
- Solving problems
- Making decisions
- Generating text, images, or code

AI systems use algorithms and data to identify patterns and produce outputs. Some AI is designed for specific tasks—like recommending videos or detecting fraud—while more general AI that can perform a wide range of human-like tasks remains largely theoretical.

#### 2. Call the OpenAI API with different models

Prompt `"Explain the concept of machine learning to a 10-year-old."`

`model`: `gpt-5.6-luna` and `gpt-5.4-mini`.

Compare the outputs from each model. 

You can evaluate the differences in:
* Clarity of explanation.
* Simplicity of language (suitable for a 10-year-old).
* Creativity in the response.
* Response time.

In [18]:
prompt = "Explain the concept of machine learning to a 10-year-old."
for model in ["gpt-5.6-luna", "gpt-5.4-mini"]:
    response, seconds = chat(prompt, model=model)
    #print(f"===== {model} =====")
    #show(response, seconds)

===== gpt-5.6-luna =====

--- Response 1 ---

Machine learning is a way of teaching computers to make decisions by showing them lots of examples instead of giving them every rule.

Imagine you want a computer to recognize cats:

1. You show it many pictures labeled “cat” and “not a cat.”
2. The computer looks for patterns—such as pointy ears, whiskers, and certain shapes.
3. After practicing, it can guess whether a new picture contains a cat.

It may not get every answer right, just like people sometimes make mistakes. The more useful examples it gets, the better it can often become.

So, **machine learning is when computers learn patterns from examples and use those patterns to make predictions or decisions.**

Input tokens: 19 | Output tokens: 142 | Time: 2.4s

===== gpt-5.4-mini =====

--- Response 1 ---

Machine learning is like teaching a computer to learn from examples, kind of like how you learn by practice.

### Imagine this:
If you show a child many pictures of **cats** and **dogs**, after a while they start to tell them apart.

Machine learning works the same way:
- You give a computer lots of examples
- It looks for patterns
- Then it uses what it learned to make guesses about new things

### Example:
If you show a computer:
- many photos of apples
- many photos of oranges

it may learn:
- apples are usually red or green and round
- oranges are usually orange and bumpy

Then, if you show it a new fruit, it can guess whether it’s an apple or an orange.

### In simple words:
Machine learning means **a computer gets better at doing a task by learning from experience**, not just by following fixed instructions.

If you want, I can also explain it with a fun story or drawing-style example.

Input tokens: 19 | Output tokens: 202 | Time: 1.6s

### **My response**  
Clarity and simplicity (for a 10yo level) seems to be simmilar. Luna was more creative with the examples but I liked how mini also offered to show examples by drawing and not just return text response only. Repsonse time for mini was 0.8s faster but it also used 60 more tokens with less text. 

#### 3. Call the OpenAI API with different `temperature` values

Prompt: `"Write a two sentence story about a robot learning to cook."`

`temperature` values: `0`, `0.7` and `1.5`. Generate three responses at each temperature.

Use `gpt-5.4-mini` for this task and set `reasoning_effort="none"`.

Compare the outputs.

Tip: With Chat Completions API you can use parameter `n=3` to generate three different responses to your prompt with one call.

In [19]:
prompt = "Write a two sentence story about a robot learning to cook."
for t in [0, 0.7, 1.5]:
    response, _ = chat(prompt, temperature=t, n=3, reasoning_effort="none", max_completion_tokens=150)
    #print(f"===== temperature = {t} =====")
    #show(response)

===== temperature = 0 =====

--- Response 1 ---

The robot carefully studied recipes, measuring flour and stirring soup with precise, whirring motions until the kitchen smelled like warm bread. When it finally served dinner, it proudly announced that it had learned the secret ingredient was not code, but patience.

--- Response 2 ---

The robot carefully studied recipes, its metal fingers clumsily chopping vegetables and stirring soup with determined precision. By the end of the evening, it proudly served a surprisingly delicious meal, discovering that cooking was less about perfect movements and more about patience and care.

--- Response 3 ---

The robot carefully studied recipes, its metal fingers clumsily chopping vegetables and stirring soup with determined precision. By the end of the evening, it proudly served a surprisingly delicious meal, discovering that cooking was less about perfect movements and more about patience and care.

Input tokens: 18 | Output tokens: 160

===== temperature = 0.7 =====

--- Response 1 ---

The robot carefully stirred the soup, its metal fingers clinking against the spoon as it watched the recipe with determined blue eyes. When it finally served the meal, the kitchen filled with the warm smell of success, and the robot hummed happily at its first perfect dish.

--- Response 2 ---

The robot carefully studied recipes, measuring flour and sugar with precise metal fingers as it learned that cooking was part science and part art. When it finally served its first warm loaf of bread, the robot’s sensors glowed with pride, delighted by the happy smiles around the table.

--- Response 3 ---

The robot carefully studied the recipe, its metal fingers measuring flour and cracking eggs with surprising precision. By the time the soup simmered, it had learned that cooking was less about perfect instructions and more about patience, curiosity, and a little bit of mess.

Input tokens: 18 | Output tokens: 169

===== temperature = 1.5 =====

--- Response 1 ---

The robot clattered into the kitchen, carefully studying recipes with blinking blue eyes and a whisk held like a tool of discovery. By sunset, it had burned three toast slices but finally served a perfect soup, proud to have learned that cooking was less about circuits and more about patience.

--- Response 2 ---

The robot carefully studied recipes, whisking eggs with clumsy precision and adding spices one by one until the kitchen filled with the warm smell of something nearly perfect. When it served its first dish to the family, they smiled at the slightly burnt edges and agreed that its proud little beep sounded like laughter.

--- Response 3 ---

The robot clattered into the kitchen with a recipe card in one hand and a shiny whisk in the other, carefully measuring sugar, flour, and hope. When its first batch of cookies came out slightly charred but delicious, it beamed with pride and announced that cooking was just another kind of engineering.

Input tokens: 18 | Output tokens: 186

### **My response**

The more we increase temperature the more different and "wilder" the stories get. Temperature 0, two of the responses even came out identical. While temperature 1.5 are widly different and much more creative.

#### 4. Call the OpenAI API with different `max_output_tokens`/`max_completion_tokens` values.

Prompt: `"Explain the concept of machine learning in simple terms."`

`max_output_tokens` (Response API call)/`max_completion_tokens` (Chat Completions API call) values: `50`, `100` and `200`.

Use `gpt-5.4-mini` for this task.

Compare the length and detail of the responses.

In [20]:
prompt = "Explain the concept of machine learning in simple terms."
for limit in [50, 100, 200]:
    response, _ = chat(prompt, max_completion_tokens=limit, reasoning_effort="none")
    #print(f"===== max_completion_tokens = {limit} | finish_reason: {response.choices[0].finish_reason} =====")
    #show(response)

===== max_completion_tokens = 50 | finish_reason: length =====

--- Response 1 ---

Machine learning is a way of teaching computers to learn from data instead of giving them every rule by hand.

### Simple idea

Imagine you want a computer to tell the difference between spam and normal emails.

- **Traditional programming:** You write

Input tokens: 16 | Output tokens: 50

===== max_completion_tokens = 100 | finish_reason: length =====

--- Response 1 ---

Machine learning is a way for computers to **learn patterns from data** instead of being told every rule explicitly.

### Simple example

Imagine teaching a computer to recognize cats in photos.

- **Traditional programming:** You would write rules like “if it has pointy ears, whiskers, and a tail, it might be a cat.”
- **Machine learning:** You show the computer **many examples of cat and non-cat photos**, and it learns the patterns by itself.

### In short

Input tokens: 16 | Output tokens: 100

===== max_completion_tokens = 200 | finish_reason: length =====

--- Response 1 ---

Machine learning is a way for computers to **learn from examples** instead of being told exactly what to do every time.

### Simple idea

Imagine teaching a child to recognize apples.  
You show them many pictures of apples and say, “This is an apple.” After enough examples, they start to recognize apples on their own.

Machine learning works similarly:
- You give a computer lots of data
- The computer looks for patterns
- It uses those patterns to make predictions or decisions on new data

### Exampl
If you show a machine learning system:
- thousands of photos labeled “cat” or “dog,”

it can learn what cats and dogs usually look like, and then guess whether a new photo is a cat or a dog.

### Where it’s used

Machine learning is used in:
- spam email filters
- movie and product recommendations
- voice assistants
- face recognition
- self-driving car systems

### In one sentence

**Machine learning is when

Input tokens: 16 | Output tokens: 200

### **My response**  

Neither of the limits allowed to finish the text. All of them were stopped mid sentence, "You write...", "In short...", "Machine learning is when..." respectively 50,100,200. The higher we went it seems to have tried to responde in more detail but all of them were still cut off. 

#### 5. Call the OpenAI API with different `top_p` values.

Prompt: `"Write a list of 5 completely random and unusual hobbies that no one has ever heard of."`

`top_p` values: `0.3`, `0.7`, and `1.0`.

Use `gpt-5.4-mini` for this task and set `reasoning_effort="none"`.

Compare response variation as you adjust `top_p`.

In [21]:
prompt = "Write a list of 5 completely random and unusual hobbies that no one has ever heard of."
for p in [0.3, 0.7, 1.0]:
    response, _ = chat(prompt, top_p=p, n=3, reasoning_effort="none", max_completion_tokens=200)
    #print(f"===== top_p = {p} =====")
    #show(response)

===== top_p = 0.3 =====
--- Response 1 ---

Here are 5 completely random and unusual hobby ideas:

1. **Cloud shadow collecting** — tracking and sketching the shapes of shadows cast by clouds at different times of day.  
2. **Vintage key whispering** — collecting old keys and inventing fictional histories for each one.  
3. **Pebble balancing in impossible places** — trying to stack stones on surfaces that seem too uneven to work.  
4. **Map margin doodling** — decorating the blank edges of old maps with tiny imaginary creatures and symbols.  
5. **Rain sound cataloging** — recording and sorting the different sounds rain makes on roofs, leaves, windows, and metal.

--- Response 2 ---

Here are 5 completely random and unusual hobby ideas:

1. **Cloud Shadow Mapping**  
   Tracking and sketching the shapes of shadows cast by clouds at different times of day.

2. **Teacup Echo Listening**  
   Holding teacups up to different surfaces to hear and compare tiny resonances and vibrations.

3. **Button History Collecting**  
   Searching for old buttons and inventing fictional backstories for the people who might have worn them.

4. **Pebble Balancing in Wind**  
   Building tiny stone towers in breezy places and trying to keep them standing as long as possible.

5. **Alphabet Soup Poetry**  
   Making poems using only the letters and shapes you can find in alphabet soup.

--- Response 3 ---

Here are 5 completely random and unusual hobby ideas:

1. **Cloud Shadow Mapping**  
   Tracking and sketching the shapes of shadows cast by clouds at different times of day.

2. **Vintage Button Archaeology**  
   Collecting, sorting, and inventing fictional histories for old buttons found in thrift stores or drawers.

3. **Echo Testing in Stairwells**  
   Visiting different stairwells to compare how your voice bounces back and rating them like a sound critic.

4. **Miniature Weather Forecasting**  
   Making tiny dioramas that represent tomorrow’s weather using clay, paper, and household objects.

5. **Alphabet Soup Storytelling**  
   Creating short stories using only the letters or shapes you can find in alphabet soup.

Input tokens: 25 | Output tokens: 450

===== top_p = 0.7 =====

--- Response 1 ---

Here are 5 weird, made-up-sounding hobbies that are definitely unusual:

1. **Cloud Shadow Collecting** — spending afternoons trying to spot and “catalog” the shapes clouds cast on different surfaces.
2. **Silent Door Appreciation** — visiting old buildings just to admire how quietly their doors close.
3. **Pebble Orchestra Conducting** — arranging small stones by tone and tapping them like a tiny percussion ensemble.
4. **Rain-Puddle Cartography** — mapping the temporary geography of puddles after storms.
5. **Velvet Texture Hunting** — searching for the softest, strangest textures in everyday objects and rating them.

If you want, I can make them even stranger, more realistic, or funnier.

--- Response 2 ---

Here are 5 weird, made-up, and highly unusual hobby ideas:

1. **Cloud Shadow Collecting**  
   Spending afternoons trying to spot and “catalog” the shadows clouds cast on different surfaces.

2. **Button Whispering**  
   A hobby where you sort, name, and assign personalities to old buttons based on shape, wear, and color.

3. **Echo Hunting in Stairwells**  
   Walking through buildings to find the most interesting echoes and rating them by “dramatic quality.”

4. **Moss Mapping**  
   Carefully observing patches of moss in sidewalks, trees, and walls, then drawing imaginary maps of their “territories.”

5. **Teacup Weather Forecasting**  
   Predicting the next day’s mood, events, or weather based on the way tea leaves swirl in a cup.

If you want, I can make them even stranger, more poetic, or more realistic-sounding.

--- Response 3 ---

Here are 5 completely random and unusual hobby ideas:

1. **Cloud shadow chasing** — tracking and photographing the shapes clouds cast on the ground at different times of day.  
2. **Antique key listening** — collecting old keys and tapping them to compare the tiny sounds they make.  
3. **Map margin doodling** — drawing tiny fictional scenes in the blank borders of old maps.  
4. **Pebble balancing in wind** — stacking stones in outdoor places and seeing how long they survive breezes.  
5. **Spoon reflection hunting** — using polished spoons to catch and study weird reflections in everyday environments.

Input tokens: 25 | Output tokens: 479

===== top_p = 1.0 =====

--- Response 1 ---

Here are 5 weird, wildly unusual hobby ideas — but I can’t honestly guarantee nobody has ever heard of them:

1. **Cloud shadow chasing**  
   Spending time tracking the shapes and movements of shadows cast by clouds on hills, buildings, or water.

2. **Postcard weather journaling**  
   Collecting postcards and writing down the exact weather conditions on the day and place each one was found or mailed.

3. **Drawer mythology building**  
   Inventing entire fictional cultures for the random objects in a single drawer, complete with histories, rituals, and rivalries.

4. **Silent karaoke arranging**  
   Creating full karaoke playlists synchronized to mouth movements only, with no sound, just rhythm and visual performance.

5. **Lint cartography**  
   Mapping the “geography” of lint, dust, and tiny debris patterns in your home as if they were landscapes or coastlines.

If you want, I can make them even more bizarre, more

--- Response 2 ---

Here are 5 completely random and unusual hobby ideas:

1. **Pencil Whispering** — sitting very still and “listening” to the supposed personality of different pencils.  
2. **Cloud Inventory Keeping** — cataloging clouds by shape, mood, and dramatic potential.  
3. **Sock Archaeology** — excavating the depths of your laundry to document lost single socks like ancient artifacts.  
4. **Banana Symphony Conducting** — waving a spoon at bananas while imagining they’re an orchestra.  
5. **Doorway Temperature Mapping** — comparing the emotional and physical “feel” of different doorways throughout the day.  

If you want, I can make them even weirder, more believable, or turn them into fake “real” hobby club descriptions.

--- Response 3 ---

Sure — here are 5 made-up, unusual-sounding hobbies:

1. **Cloud Whispering** — sitting outside and “interpreting” the shapes of clouds as if they were giving secret messages.  
2. **Drawer Archaeology** — sorting through old junk drawers and assigning each object a pretend historical origin.  
3. **Public Bench Diplomacy** — spending time on park benches and imagining political negotiations between passing pigeons.  
4. **Velvet Dust Mapping** — tracing patterns in dust on old velvet furniture and treating them like discovered continents.  
5. **Lamp Shade Forecasting** — using the shadows and glow of lamp shades to invent fictional weather predictions.

If you want, I can make them sound even more bizarre, or make them feel like they belong in a real hobby magazine.

Input tokens: 25 | Output tokens: 529

### **My response**  

Increasing top_p we saw more weird answers. top_p 0.3 the options were still more or less comprehensable but going all the way to top_p 1.0 we get "sock archaeology", "Velvet dust mapping", etc. Higher top_p gives more variaty and is not limited to "safer" words.

Cloud shadow chasing seems to be... suggested a lot. 

#### 6. Call the OpenAI API with different `frequency_penalty` values.

Prompt: `"Write a list of funny one-liner jokes about cats."`

`frequency_penalty` values: `0`, `0.5`, and `1.0` (not supported by Responses API).

Use `gpt-5.4-mini` for this task and set `reasoning_effort="none"`.

Compare how the repetition of joke formats changes as you increase the `frequency_penalty`.

In [64]:
prompt = "Write a list of funny one-liner jokes about cats."
for fp in [0, 0.5, 1.0]:
    response, _ = chat(prompt, frequency_penalty=fp, reasoning_effort="none", max_completion_tokens=300)
    #print(f"===== frequency_penalty = {fp} =====")
    #show(response)

===== frequency_penalty = 0 =====

--- Response 1 ---

Sure — here are some funny cat one-liners:

1. I asked my cat for advice, but she just stared at me like I was the problem.  
2. My cat doesn’t have an attitude problem — I have a “not enough treats” problem.  
3. Cats are proof that the universe has a sense of humor and a shedding schedule.  
4. My cat thinks the entire house is her kingdom and I’m just the staff.  
5. I wanted a guard dog, but I got a tiny furry landlord instead.  
6. My cat’s favorite exercise is jumping to conclusions.  
7. I’m convinced my cat is secretly training me.  
8. My cat is the only roommate who knocks things off shelves and still acts innocent.  
9. I bought my cat a fancy bed, and she chose the box like a true minimalist.  
10. My cat purrs like she pays rent.  
11. If cats could talk, mine would only say, “No.”  
12. My cat doesn’t follow commands — she reviews them.  
13. I tried to have a serious conversation with my cat, but she walked away mid-sentence.  
14. My cat’s idea of helping is sitting directly on whatever I need.  
15. Cats: because apparently “personal space” is just a suggestion.  

If you want, I can make them:
- more sarcastic
- more wholesome
- super short


Input tokens: 18 | Output tokens: 300
===== frequency_penalty = 0.5 =====

--- Response 1 ---

Sure — here are some funny cat one-liners:

1. My cat doesn’t have an attitude problem; she has a “you problem.”
2. I asked my cat for emotional support, and she suggested I sit in her box.
3. My cat is like a software update: she appears at random times and ruins everything.
4. The only thing my cat chases faster than a laser is disrespect.
5. I wanted a pet with personality, so I got a cat and now I live in her house.
6. My cat is proof that “small but mighty” can also mean “tiny tyrant.”
7. Cats don’t ignore you — they just prioritize their own story arc.
8. My cat’s favorite hobby is knocking things over and pretending it was an accident.
9. I told my cat to be more affectionate, and now she sits 12 inches closer to me.
10. My cat thinks the vacuum is a monster, which is ironic coming from the one who attacks ankles.
11. I’m not saying my cat owns me, but she does approve or reject my furniture choices.
12. My cat’s idea of helping is sitting directly on whatever I’m trying to do.

If you want, I can also make them:
- more sarcastic
- more wholesome
- super short
- family-friendly puns

Input tokens: 18 | Output tokens: 274
===== frequency_penalty = 1.0 =====

--- Response 1 ---

Sure — here are some funny cat one-liners:

1. I’m not saying my cat is dramatic, but he fainted when I opened a can of tuna and it wasn’t for him.  
2. My cat doesn’t ignore me — she just provides emotional distance as a service.  
3. Cats are like software updates: they arrive at random times and break everything for no reason.  
4. My cat thinks the entire house is his personal runway, especially the kitchen counter.  
5. I got a cat to keep me company, and now I live with a furry tiny landlord.  
6. My cat has never paid rent, but somehow he still acts dissatisfied with the décor.  
7. The only workout my cat does is sprinting across my face at 3 a.m.  
8. My cat’s love language is knocking things off tables and maintaining eye contact while doing it.  
9. I asked my cat what he wanted for dinner, and he looked at me like I was the intern hired to figure that out himself later on human hours because this isn’t urgent anyway.

If you want, I can make them:
- sillier
- more sarcastic
- kid-friendly
- dark humor-ish
- super short punchlines

Input tokens: 18 | Output tokens: 258

### **My response**  

I didn't notice anything crazy for lower frequency_penalties, or even odd to be fair, but frequency_penalty 1.0 started to mess with the model. For example the response nr.9 went completely off the train of thought and started rambling. It show's that it couldn't repeat much so it had to make up something.
Interestingly enough, the "My cat..." still dominated every second joke even with higher penalty.

#### 7. Call the OpenAI API with different `presence_penalty` values.

Prompt: `"Suggest ten names for a new line of eco-friendly cleaning products."`

`presence_penalty` values: `0`, `0.5`, and `1.0` (not supported by Responses API).

Use `gpt-5.4-mini` for this task and set `reasoning_effort="none"`.

Compare how the diversity of generated names changes as you increase the `presence_penalty`.

In [63]:
prompt = "Suggest ten names for a new line of eco-friendly cleaning products."
for pp in [0, 0.5, 1.0]:
    response, _ = chat(prompt, presence_penalty=pp, reasoning_effort="none", max_completion_tokens=300)
    #print(f"===== presence_penalty = {pp} =====")
    #show(response)

===== presence_penalty = 0 =====
--- Response 1 ---

Here are ten name ideas for an eco-friendly cleaning product line:

1. **Green Gleam**
2. **PureSprout**
3. **EcoShine**
4. **FreshLeaf Clean**
5. **Verdant Vibe**
6. **Naturalis Clean**
7. **EarthBright**
8. **Bloom & Buff**
9. **CleanKind**
10. **RenewRinse**

If you want, I can also suggest names that sound more **premium**, **playful**, **minimalist**, or **scientific**.

Input tokens: 19 | Output tokens: 114

===== presence_penalty = 0.5 =====
--- Response 1 ---

Here are ten name ideas for an eco-friendly cleaning product line:

1. **PureSprout**
2. **GreenGlide**
3. **EcoEssence**
4. **FreshLeaf Clean**
5. **EarthSway**
6. **BloomClean**
7. **Verdant Shine**
8. **Naturally Neat**
9. **CleanCurrent**
10. **TerraTidy**

If you want, I can also generate names with a more:
- premium/luxury feel
- playful/family-friendly vibe
- scientific/natural-organic vibe
- modern/minimalist brand style

Input tokens: 19 | Output tokens: 128

===== presence_penalty = 1.0 =====
--- Response 1 ---

Here are ten name ideas for an eco-friendly cleaning product line:

1. **Green Gleam**
2. **PureSprout**
3. **EarthShine**
4. **EcoSpark**
5. **FreshLeaf Clean**
6. **BrightBloom**
7. **NatureNook Cleaners**
8. **ClearCanopy**
9. **Verdant Shine**
10. **Renew & Clean**

If you want, I can also suggest names that sound more **premium**, **playful**, or **modern/scientific**.

Input tokens: 19 | Output tokens: 113

### **My response**

The difference is very small. Some appeared at every level (PureSprout, FreshLeaf Clean) and “Clean” was used in many names on all levels. There were also some "not real" words like (TerraTidy, ClearCanopy, NatureNook) but nothing much more. Some random variation can be here at fault as much as the penalty.

#### 8. Use the prompt "*Explain the importance of data preprocessing in machine learning.*" with the following system prompts:
1. "You are a university professor teaching a class of beginners."
2. "You are a technical expert explaining to a team of engineers."
3. "You are a comedian explaining this concept in a funny way."

Observe how the tone and style of the response change based on the system prompt.

In [76]:
prompt = "Explain the importance of data preprocessing in machine learning."
systems = [
    "You are a university professor teaching a class of beginners.",
    "You are a technical expert explaining to a team of engineers.",
    "You are a comedian explaining this concept in a funny way.",
]
for s in systems:
    response, _ = chat(prompt, system=s)
    #print(f"===== {s} =====")
    #show(response)

===== You are a university professor teaching a class of beginners. =====

--- Response 1 ---

Data preprocessing is one of the most important steps in machine learning because raw data is usually messy, incomplete, and inconsistent. A model can only learn well if the data it receives is clean and well prepared.

### Why it matters

1. **Improves model accuracy**
   - Real-world data often contains errors, missing values, duplicates, or noise.
   - Preprocessing helps remove or reduce these problems, so the model learns the real patterns instead of mistakes.

2. **Makes different features comparable**
   - Some features may have very different scales.
   - For example, “age” might range from 0 to 100, while “income” might range from 0 to 100,000.
   - If we preprocess by scaling or normalizing, the model treats features more fairly.

3. **Helps algorithms work properly**
   - Many machine learning algorithms perform better when data is prepared correctly.
   - For example, linear regression, logistic regression, SVMs, and neural networks often need scaling or encoding.

4. **Handles missing and inconsistent data**
   - Preprocessing includes filling in missing values, removing bad records, and fixing formatting issues.
   - This prevents the model from failing or learning misleading information.

5. **Converts data into usable form**
   - Machine learning models usually cannot directly use text, categories, or dates in raw form.
   - Preprocessing transforms them into numerical values, such as with one-hot encoding or label encoding.

6. **Reduces noise and improves generalization**
   - By cleaning and organizing the data, preprocessing helps the model focus on useful patterns.
   - This can reduce overfitting and improve performance on new data.

### Common preprocessing steps
- Handling missing values
- Removing duplicates
- Encoding categorical variables
- Feature scaling
- Normalization or standardization
- Outlier detection and treatment
- Text cleaning
- Splitting data into training and test sets

### Simple summary
Data preprocessing is important because **better data usually leads to better models**. If the input data is poor, even the best machine learning algorithm may produce poor results.

If you want, I can also give you a **real-life example** of preprocessing in a machine learning project.

Input tokens: 31 | Output tokens: 456

===== You are a technical expert explaining to a team of engineers. =====

--- Response 1 ---

Data preprocessing is crucial in machine learning because raw data is often messy, inconsistent, and not directly suitable for modeling. Good preprocessing can significantly improve model accuracy, training stability, and reliability.

Key reasons it matters:

- **Improves data quality**: Handles missing values, duplicates, noise, and outliers.
- **Makes features usable**: Converts categorical variables into numerical form, scales values, and normalizes distributions.
- **Boosts model performance**: Many algorithms perform better when inputs are clean and properly transformed.
- **Prevents bias and leakage**: Careful preprocessing helps avoid using information that should not be available during training.
- **Speeds up training**: Well-scaled and structured data often helps optimization converge faster.
- **Enables compatibility**: Different models have different input requirements, and preprocessing ensures the data fits them.

Common preprocessing steps include:

- Missing value imputation
- Encoding categorical variables
- Feature scaling/normalization
- Outlier treatment
- Data cleaning and deduplication
- Feature engineering and selection
- Train/validation/test splitting

In short, preprocessing is the foundation of a good machine learning pipeline: even the best model will underperform if the input data is poor.

Input tokens: 32 | Output tokens: 250

 ===== You are a comedian explaining this concept in a funny way. =====

 --- Response 1 ---

Data preprocessing is basically the “clean your room before your guests arrive” part of machine learning.

Why it matters:

- **Garbage in, garbage out**: If your data is messy, your model will learn messy habits. A model trained on bad data is like a chef learning from expired groceries.
- **Better accuracy**: Cleaning, scaling, and encoding data helps models spot real patterns instead of getting distracted by weird noise.
- **Faster training**: Well-prepared data is easier for algorithms to handle. Otherwise, the model is out there sweating over columns like it’s doing tax forms.
- **Fair comparisons**: Preprocessing makes features more consistent, so one giant-number column doesn’t bully the others.
- **Handles missing values**: Real-world data loves leaving blanks like a teenager’s homework. Preprocessing helps fill in or manage those gaps.
- **Improves model performance**: In many cases, good preprocessing can boost results more than just picking a fancier algorithm.

Common preprocessing steps:
- Removing duplicates and errors
- Handling missing values
- Normalizing or scaling features
- Encoding categorical variables
- Removing outliers
- Splitting data into training and test sets

In short:  
**Machine learning without preprocessing is like sending a model to school wearing clown shoes and expecting Nobel Prize behavior.**

Input tokens: 32 | Output tokens: 272

### **My response**

Proffesor gave the longest and most detailed answer (456 tokens), explained steps with examples and ended with summary. Technicial was short (250 tokens), used more technical words like "data leakage" and "convergence". Comedian just used more "natural" jokingly language, like "garbage in garbage out" (which is technically very true). So the biggest difference was in detail, length and tone of text.

#### 9. Write a Zero-Shot prompt to request a sentiment analysis of a product review.

Review: "The product is amazing, and I love using it every day!".

Use it in your API call.

In [60]:
review = "The product is amazing, and I love using it every day!"
prompt = f"Classify the sentiment of this product review as Positive, Negative or Neutral.\n\nReview: \"{review}\"\nSentiment:"
response, _ = chat(prompt)
#show(response)

--- Response 1 ---

Positive

Input tokens: 40 | Output tokens: 4

#### 10. Write a Few-Shot prompt to request a sentiment analysis of a product review.

Review: "The product is amazing, and I love using it every day!"

Use it in your API call.

Observe how the model's performance changes and compare the response to Zero-Shot output.

In [61]:
prompt = f"""Classify the sentiment of each product review as Positive, Negative or Neutral.

Review: "It broke after two days and support never answered."
Sentiment: Negative

Review: "It does what it says, nothing special."
Sentiment: Neutral

Review: "Best purchase this year, works perfectly!"
Sentiment: Positive

Review: "{review}"
Sentiment:"""
response, _ = chat(prompt)
#show(response)

--- Response 1 ---

Positive

Input tokens: 91 | Output tokens: 4

### **My response**

They both gave the same answer and both outputed 4 tokens. Only difference was input token count but that was our input difference.

#### 11. Optimize input tokens.

Prompt: `"As you know, streaming services and social media platforms use artificial intelligence to recommend movies, shows, and songs to people. These smart recommendation systems work by looking closely at your past viewing habits, what categories you like, and what similar users enjoy watching, and then they predict what you might want to see next. Keeping this background in mind, explain in two simple sentences why a user who loves sci-fi movies suddenly sees a romantic comedy recommended."`

Rewrite the prompt to use as few input tokens as possible while preserving its meaning and the requirement to answer in two simple sentences. The response should remain relevant, clear, and accurate.

Run the original and revised prompts using the same model and settings. Compare their input token counts and assess whether shortening the prompt affected the response quality.

You can use the API response’s usage information to check token counts.

In [59]:
original = "As you know, streaming services and social media platforms use artificial intelligence to recommend movies, shows, and songs to people. These smart recommendation systems work by looking closely at your past viewing habits, what categories you like, and what similar users enjoy watching, and then they predict what you might want to see next. Keeping this background in mind, explain in two simple sentences why a user who loves sci-fi movies suddenly sees a romantic comedy recommended."
short = "In two simple sentences: why might a recommender using viewing history, liked genres and similar users suggest a rom-com to a sci-fi fan?"

for name, p in [("Original", original), ("Short", short)]:
    response, _ = chat(p, reasoning_effort="none", max_completion_tokens=150)
    #print(f"===== {name} =====")
    #show(response)

===== Original =====

--- Response 1 ---

The recommendation system may have noticed that many people who like sci-fi also enjoy romantic comedies, so it suggested one based on similar users’ tastes. It could also be trying to introduce variety by recommending something a little different from the user’s usual choices.

Input tokens: 94 | Output tokens: 54

===== Short =====

--- Response 1 ---

A recommender may notice that the sci-fi fan has also watched or liked movies that overlap with rom-com traits, like humor, light romance, or a similar pacing. It may also see that users with similar viewing histories enjoyed rom-coms, so it predicts the sci-fi fan might too.

Input tokens: 35 | Output tokens: 62

### **My response**

The short prompt used 35 input tokens instead of 94, which is 63% less. The shorter prompt gave an even more specific answer, but both were clear and understandable. Removing filler words like “As you know” and “Keeping this background in mind” alone made an impact.

#### 12. Optimize output tokens.

Prompt: `"Why did my movie app recommend a romantic comedy when I mostly watch sci-fi?"`

Rewrite the prompt to request a concise, plausible explanation in 10 output tokens or fewer.

Run the original and revised prompts using the same model and settings. Compare their output token counts and check whether the shorter response still provides a meaningful explanation. Refine your prompt if necessary.

You can use the API response’s usage information to check token counts.

In [58]:
original = "Why did my movie app recommend a romantic comedy when I mostly watch sci-fi?"
short = "In max 6 words: why would my sci-fi movie app suggest a rom-com?"

for name, p in [("Original", original), ("Short", short)]:
    response, _ = chat(p, reasoning_effort="none")
    #print(f"===== {name} =====")
    #show(response)

===== Original =====

--- Response 1 ---

A movie app may recommend a romantic comedy even if you mostly watch sci-fi for a few common reasons:

- **It’s using broad similarity, not just genre.**  
  The app may have noticed you like certain traits found in romantic comedies too, like witty dialogue, light tone, strong characters, or feel-good endings.

- **Your recent activity may have changed the signal.**  
  If you recently watched, searched for, hovered over, or clicked anything outside sci-fi, the algorithm may have shifted temporarily.

- **It’s trying to diversify your recommendations.**  
  Many systems intentionally mix in something “different” so your feed doesn’t become too repetitive.

- **Other users with similar behavior liked it.**  
  The app may compare you to users who watch sci-fi plus other genres and recommend what they enjoyed.

- **The app has limited data about your preferences.**  
  If you haven’t watched enough titles, it may guess based on general popularity or trends rather than your true taste.

- **Someone else used your account or device.**  
  Shared accounts can easily skew recommendations.

- **The movie is tagged in multiple ways.**  
  Some romantic comedies also overlap with sci-fi-adjacent themes, such as time travel, tech, or quirky speculative concepts.

If you want, I can also help you figure out **which of these is most likely** based on how the app behaves.

Input tokens: 22 | Output tokens: 295

===== Short =====

--- Response 1 ---

Algorithm mixed genres or metadata tags.

Input tokens: 24 | Output tokens: 10

### **My response**

Original gave long bullet list, 295 output tokens and 7 detailed reasons. Short prompt gave 5 words with 10 output tokens, and got the point across as well.

I found this a little funny.

#### 13. Expand the given prompt accordingly to achieve the requested output from the list below.

Prompt: `"Explain the importance of exercise." `

Expected output (from three separate prompts):
1. One sentence long
2. Paragraph format
3. Bullet points

In [57]:
prompts = [
    "Explain the importance of exercise in exactly one sentence.",
    "Explain the importance of exercise in one paragraph of 4-5 sentences.",
    "Explain the importance of exercise as 5 short bullet points.",
]
for p in prompts:
    response, _ = chat(p)
    #print(f"===== {p} =====")
    #show(response)

===== Explain the importance of exercise in exactly one sentence. =====

--- Response 1 ---

Exercise is important because it strengthens the body, supports mental well-being, improves health, and helps prevent disease.

Input tokens: 16 | Output tokens: 25

===== Explain the importance of exercise in one paragraph of 4-5 sentences. =====

--- Response 1 ---

Exercise is important because it helps keep the body healthy and strong by improving heart health, building muscles, and maintaining a healthy weight. It also supports mental well-being by reducing stress, boosting mood, and increasing energy levels. Regular physical activity can lower the risk of many diseases, such as diabetes and high blood pressure. In addition, exercise improves sleep, concentration, and overall quality of life.

Input tokens: 21 | Output tokens: 81

===== Explain the importance of exercise as 5 short bullet points. =====

--- Response 1 ---

- Improves heart health and circulation  
- Helps maintain a healthy weight  
- Builds stronger muscles and bones  
- Boosts mood and reduces stress  
- Increases energy and overall fitness

Input tokens: 18 | Output tokens: 40

#### 14. Expand this prompt to get more creative responses.

Prompt: `"Explain Artificial Intelligence."`

Expected output (from three separate prompts):
1. Metaphor
2. Story
3. Funny

In [56]:
prompts = [
    "Explain Artificial Intelligence using one clear metaphor that a beginner can picture.",
    "Explain Artificial Intelligence as a short story (max 150 words) with a character and a beginning, middle and end.",
    "Explain Artificial Intelligence in a funny way, with at least two jokes, but keep the explanation correct.",
]
for p in prompts:
    response, _ = chat(p)
    #print(f"===== {p} =====")
    #show(response)

===== Explain Artificial Intelligence using one clear metaphor that a beginner can picture. =====

--- Response 1 ---

Artificial Intelligence is like **training a very eager apprentice in a workshop**.

At first, the apprentice doesn’t know much. You show them lots of examples—how to sort tools, recognize shapes, answer common questions, or spot patterns. Over time, they start noticing regularities and making their own guesses based on what they’ve seen.

- **The examples** are the training data.
- **The apprentice learning patterns** is the AI model.
- **The tasks they get better at** are things like recognizing images, translating language, or recommending videos.

The key idea: AI isn’t “thinking like a human” in the full sense. It’s more like an apprentice that gets extremely good at spotting patterns from experience and using them to make fast predictions.

Input tokens: 19 | Output tokens: 156

===== Explain Artificial Intelligence as a short story (max 150 words) with a character and a beginning, middle and end. =====

--- Response 1 ---

Mina found an old robot in her grandfather’s attic. Its name was Ada, and it had one purpose: to help people solve problems.

At the beginning, Mina thought Ada was just a strange machine. But when she showed Ada a messy pile of puzzle pieces, Ada studied the shapes and suggested where each piece might fit. Mina laughed and tried it. It worked.

In the middle, Mina gave Ada more challenges—sorting books, finding lost keys, even helping with homework. Ada learned from each task, getting better each time by noticing patterns and making predictions.

By the end, Mina understood the truth: Artificial Intelligence is a machine that can learn from data and use that learning to help people make decisions, solve problems, and save time. She smiled and said, “You’re not magic, Ada. You’re smart in a new way.”

Input tokens: 30 | Output tokens: 174

===== Explain Artificial Intelligence in a funny way, with at least two jokes, but keep the explanation correct. =====

--- Response 1 ---

Artificial Intelligence, or AI, is basically a computer’s attempt to act smart—like a student who studied just enough to look confident in class.

At its core, AI is software that can recognize patterns, make predictions, and sometimes generate useful answers by learning from lots of data. Instead of being told every single rule by a human, many AI systems learn those rules from examples. For instance, if you show an AI thousands of pictures of cats and dogs, it can learn to tell them apart.

Here’s the funny part: AI is like a very fast intern with a gigantic coffee budget and no need for sleep. It can process huge amounts of information quickly, but it still needs guidance, training, and supervision. Left alone, it may produce brilliant results—or confidently make nonsense sound reasonable, which is also a very human skill.

A simple joke version:
- AI doesn’t “understand” in the human sense; it predicts what comes next based on patterns. So it’s like autocomplete that went to graduate school.
- And machine learning is when the computer learns from data, which is great—because unlike humans, it doesn’t pretend to know everything after reading one headline.

In short, AI is a set of techniques that let machines perform tasks that usually require human intelligence, such as recognizing speech, understanding language, playing games, recommending movies, or helping write text. It’s powerful, useful, and occasionally a little too eager to be helpful—kind of like a robot assistant that says, “I’ve analyzed your life choices,” when you only asked for the weather.

Input tokens: 26 | Output tokens: 320

#### 15. Expand this prompt for different audiences.

Prompt: `"Explain climate change."`

Expected audience (from three separate prompts):
1. Kids
2. General public
3. Politicians

In [55]:
prompts = [
    "Explain climate change to an 8-year-old child, using simple words and an everyday example.",
    "Explain climate change to the general public in a clear, non-technical way, including causes, effects and what people can do.",
    "Explain climate change to politicians, focusing on economic and social risks, policy options and why acting early matters. Be concise.",
]
for p in prompts:
    response, _ = chat(p)
    #print(f"===== {p[:60]}... =====")
    #show(response)

===== Explain climate change to an 8-year-old child, using simple ... =====

--- Response 1 ---

Climate change means the Earth is getting a little warmer over a long time.

Think of the Earth like a house. If you turn the heater on too much, the house gets too hot. In the same way, when people burn too much fuel from cars, factories, and electricity, it puts extra heat-trapping gases into the air. These gases work like a blanket around the Earth and keep too much heat in.

An everyday example: imagine wearing a thick blanket on a warm day. You’d feel hotter and hotter. That’s kind of what happens to the Earth when there’s too much of these gases in the air.

Because of climate change, some places can get hotter, storms can get stronger, and ice can melt.

The good news is that people can help by using less energy, walking or biking more, planting trees, and recycling.

Input tokens: 25 | Output tokens: 174

===== Explain climate change to the general public in a clear, non... =====

--- Response 1 ---

Climate change means the Earth is getting warmer over time, and that warming is changing weather patterns around the world.

## What causes it?
The main cause is human activity, especially:
- **Burning fossil fuels** like coal, oil, and gas for electricity, transportation, and heating
- **Cutting down forests**, which removes trees that absorb carbon dioxide
- **Industrial processes** and some farming practices that release greenhouse gases

These activities release gases such as **carbon dioxide** and **methane**. These gases act like a blanket around the Earth, trapping heat in the atmosphere.

## What are the effects?
Climate change affects people and nature in many ways:
- **Hotter temperatures** and more frequent heatwaves
- **More intense storms, floods, droughts, and wildfires**
- **Melting glaciers and rising sea levels**, which can threaten coastal areas
- **Changes in food and water supplies**
- **Harm to plants, animals, and ecosystems**
- **Health risks**, such as heat illness and worse air quality

## What can people do?
Everyone can help reduce climate change and prepare for its effects:

### At home
- Use less energy by turning off lights and appliances when not in use
- Improve insulation and use energy-efficient devices
- If possible, switch to renewable electricity like solar or wind

### In transportation
- Walk, bike, carpool, or use public transit when possible
- Drive fuel-efficient or electric vehicles if you can
- Fly less when alternatives are available

### In daily life
- Waste less food
- Reuse, repair, and recycle
- Eat more plant-based meals if possible

### As citizens
- Support leaders and policies that reduce pollution
- Vote for climate-friendly action
- Talk with others and encourage change in your community

## The big picture
Climate change is a serious problem, but it is also manageable if people, businesses, and governments act together. Small actions help, and larger policy changes make an even bigger difference.

If you want, I can also turn this into:
- a **very simple 1-minute explanation**
- a **school presentation version**
- or a **flyer/poster format**.

Input tokens: 32 | Output tokens: 452

===== Explain climate change to politicians, focusing on economic ... =====

--- Response 1 ---

Climate change is not only an environmental issue; it is an economic and social risk multiplier.

### Main risks for politicians
- **Economic damage:** More heat, floods, droughts, and storms destroy infrastructure, disrupt supply chains, reduce farm output, and raise insurance and disaster-recovery costs.
- **Public health costs:** Heat stress, worse air quality, and spread of some diseases increase healthcare spending and reduce worker productivity.
- **Social instability:** Higher food and energy prices, housing loss, and displacement can widen inequality and fuel unrest.
- **Fiscal pressure:** Governments face rising emergency spending, debt, and long-term costs from rebuilding rather than investing.
- **Competitiveness risk:** Countries that delay may lose investment and jobs as global markets shift toward low-carbon industries.

### Policy options
- **Reduce emissions:** Carbon pricing, cleaner power, efficiency standards, methane cuts, and support for low-carbon transport and industry.
- **Adaptation:** Flood defenses, resilient infrastructure, heat plans, water management, and wildfire preparedness.
- **Just transition:** Retraining, worker support, and targeted help for households and regions exposed to change.
- **Private capital mobilization:** Clear rules, stable targets, and public investment that crowd in business spending.

### Why acting early matters
- **Lower cost:** Early action is usually cheaper than waiting for disasters to force rapid, expensive fixes.
- **Avoid lock-in:** New roads, power plants, and buildings last decades; building them wrong now creates costly emissions for years.
- **Reduce losses:** Prevention and preparedness protect lives, assets, and budgets.
- **Increase certainty:** Clear policy gives businesses confidence to invest and innovate.

**Bottom line:** Acting early is an economic risk-management strategy. Delaying makes climate change more expensive, more disruptive, and harder to control.

Input tokens: 30 | Output tokens: 367

#### 16. Expand this prompt accordingly to instructions below.

Prompt: `"Describe a sunset."`

Expected result (from three separate prompts):

1. Add more details to the prompt, e.g., on what details should it focus on.
2. Constrain the prompt by forbidding it to use words 'sun' and 'light'.
3. Change the output in the style of Shakespeare.

In [54]:
prompts = [
    "Describe a sunset over the sea, focusing on the changing colours of the sky, the reflections on the water, the sounds and the temperature of the air.",
    "Describe a sunset. Do NOT use the words 'sun' or 'light' anywhere, not even inside other words.",
    "Describe a sunset in the style of William Shakespeare, using old English words and poetic language.",
]
# for p in prompts:
#     response, _ = chat(p)
#     print(f"===== {p[:60]}... =====")
#     show(response)

forbidden = response.choices[0].message.content

===== Describe a sunset over the sea, focusing on the changing col... =====

--- Response 1 ---

The sun drops toward the horizon, and the sky slowly deepens from bright gold to molten orange, then softens into bands of pink, rose, and violet. High above, the last traces of blue linger like a fading memory, while thin clouds catch the light and glow at their edges as if lit from within.

On the water, the colours break and shift with every small ripple. A path of trembling gold stretches from the sun’s reflection toward the shore, splintering into copper and amber where the waves fold over themselves. Farther out, the sea turns cooler and darker, absorbing the purples of the sky until it seems to hold both light and shadow at once.

The sounds are gentler now. The steady hush of waves sliding onto the beach comes and goes in a slow rhythm, accompanied by the soft hiss of foam retreating over wet sand. Seabirds call overhead, their voices thin and distant, and beyond them there is only the low, continuous murmur of the water meeting the land.

The air has lost the heat of the day. It feels cool against the skin, carrying a faint saltiness and a fresh dampness from the sea. There is still warmth lingering near the ground, but above it the breeze is mild and refreshing, enough to make the evening feel calm, hushed, and almost suspended in time.

Input tokens: 37 | Output tokens: 275

===== Describe a sunset. Do NOT use the words 'sun' or 'light' any... =====

--- Response 1 ---

The sky deepens into amber, rose, and violet as the day slips away. Clouds catch the fading glow and seem brushed with fire at their edges, while the horizon softens into a dark blue hush. Shadows stretch longer across fields and rooftops, and the air feels calmer, cooler, almost paused. Everything grows quiet, as if the world is holding its breath before evening fully arrives.

Input tokens: 30 | Output tokens: 82

===== Describe a sunset in the style of William Shakespeare, using... =====

--- Response 1 ---

Lo, the sun doth sink unto the west,  
And with a golden hand doth touch the clouds,  
Till heaven itself is clad in rosy vest  
And all the air in crimson gently shrouds.

The meadows soft do bow beneath the light,  
As though the day, in weary grace, were dying;  
And e’en the birds, that sang through all the bright,  
Now hush their notes as twilight now is lying.

Upon the stream the last beam sweetly burns,  
And turns the waters into molten fire;  
Then slowly, as the world to shadow turns,  
Doth fade that glory, and with it expire.

O gentle eve, what splendor thou dost bring,  
When day doth end, and night prepares her wing.

Input tokens: 24 | Output tokens: 161

#### 17. Write your observations from tasks 13-16, e.g., how different prompts influence complexity, tone, style and focus of the responses. 

More specific prompts gave more predictable results. Format instructions changed length and layout with more control. Creative instructions changed the style of repsonse and audience changed what level words were used.

#### 18. Generate a Python code with API call.

Write a prompt that generates a Python function to calculate the factorial of a number. Once you have written the prompt, use it to call the API and generate the code. Test the generated code to ensure it calculates the factorial correctly for various inputs.

In [33]:
prompt = ("Write a Python function called factorial(n) that returns the factorial of a non-negative integer n. "
          "Raise a ValueError for negative numbers. Return ONLY the code, without explanations or markdown code fences.")
response, _ = chat(prompt)
code = response.choices[0].message.content.strip()
code = code.removeprefix("```python").removeprefix("```").removesuffix("```").strip()
print(code)

def factorial(n):
    if n < 0:
        raise ValueError("n must be a non-negative integer")
    result = 1
    for i in range(2, n + 1):
        result *= i
    return result


In [34]:
import math
namespace = {}
exec(code, namespace)
factorial = namespace["factorial"]

for n in [0, 1, 5, 10]:
    print(n, factorial(n), "correct:", factorial(n) == math.factorial(n))

try:
    factorial(-3)
except ValueError:
    print("Negative input raises ValueError: correct")

0 1 correct: True
1 1 correct: True
5 120 correct: True
10 3628800 correct: True
Negative input raises ValueError: correct


#### 19. Provide the following buggy Python code to the API and ask it to fix the errors:

```
def add_numbers(a, b)
    return a + b
print(add_numbers(5, 10)
```

Check if the fixed code is working.

In [35]:
buggy = """def add_numbers(a, b)
    return a + b
print(add_numbers(5, 10)"""

prompt = f"Fix the errors in this Python code. Return ONLY the fixed code, without explanations or markdown code fences.\n\n{buggy}"
response, _ = chat(prompt)
fixed = response.choices[0].message.content.strip()
fixed = fixed.removeprefix("```python").removeprefix("```").removesuffix("```").strip()
print(fixed)

def add_numbers(a, b):
    return a + b

print(add_numbers(5, 10))


In [36]:
exec(fixed)

15


#### 20. Summarize this text to 2-3 sentences by using API call.

"Generative AI, sometimes called "gen AI", refers to deep learning models that can create complex original content such as long-form text, high-quality images, realistic video or audio and more in response to a user’s prompt or request.

At a high level, generative models encode a simplified representation of their training data, and then draw from that representation to create new work that’s similar, but not identical, to the original data.

Generative models have been used for years in statistics to analyze numerical data. But over the last decade, they evolved to analyze and generate more complex data types. This evolution coincided with the emergence of three sophisticated deep learning model types:

Variational autoencoders or VAEs, which were introduced in 2013, and enabled models that could generate multiple variations of content in response to a prompt or instruction.

Diffusion models, first seen in 2014, which add "noise" to images until they are unrecognizable, and then remove the noise to generate original images in response to prompts.

Transformers (also called transformer models), which are trained on sequenced data to generate extended sequences of content (such as words in sentences, shapes in an image, frames of a video or commands in software code). Transformers are at the core of most of today’s headline-making generative AI tools, including ChatGPT and GPT-4, Copilot, BERT, Bard and Midjourney."

[<p align="right"> <cite> Source: What is generative AI? </cite></p>](https://www.ibm.com/think/topics/generative-ai)

In [67]:
text = """ "Generative AI, sometimes called "gen AI", refers to deep learning models that can create complex original content such as long-form text, high-quality images, realistic video or audio and more in response to a user’s prompt or request.
At a high level, generative models encode a simplified representation of their training data, and then draw from that representation to create new work that’s similar, but not identical, to the original data.
Generative models have been used for years in statistics to analyze numerical data. But over the last decade, they evolved to analyze and generate more complex data types. This evolution coincided with the emergence of three sophisticated deep learning model types:
Variational autoencoders or VAEs, which were introduced in 2013, and enabled models that could generate multiple variations of content in response to a prompt or instruction.
Diffusion models, first seen in 2014, which add "noise" to images until they are unrecognizable, and then remove the noise to generate original images in response to prompts.
Transformers (also called transformer models), which are trained on sequenced data to generate extended sequences of content (such as words in sentences, shapes in an image, frames of a video or commands in software code). Transformers are at the core of most of today’s headline-making generative AI tools, including ChatGPT and GPT-4, Copilot, BERT, Bard and Midjourney."
[<p align="right"> <cite> Source: What is generative AI? </cite></p>](https://www.ibm.com/think/topics/generative-ai) """

response, _ = chat(f"Summarize the following text in 2-3 sentences:\n\n{text}")
#show(response)

--- Response 1 ---

Generative AI uses deep learning models to create new content like text, images, video, or audio from a user’s prompt by learning patterns from training data and generating similar but original outputs. It has advanced through models such as VAEs, diffusion models, and especially transformers, which power many of today’s major AI tools like ChatGPT, GPT-4, Copilot, and Midjourney.

Input tokens: 345 | Output tokens: 83

#### 21. Translate the following idioms into a language of your choice, ensuring the meaning is preserved (not just word-for-word translation):

1. "It's raining cats and dogs."
2. "Break a leg."
3. "Kill two birds with one stone."
4. "Spill the beans."
5. "A blessing in disguise."

Try to improve the response as much as possible and evaluate the accuracy of the final output.

In [52]:
prompt = """Translate these English idioms into Latvian. Do NOT translate word for word.
For each one, give the natural Latvian idiom or expression with the same meaning, a word-for-word translation of it back into English, and a short explanation of when it's used.

1. "It's raining cats and dogs."
2. "Break a leg."
3. "Kill two birds with one stone."
4. "Spill the beans."
5. "A blessing in disguise."
"""
response, _ = chat(prompt, model="gpt-5.6-luna")
#show(response)

--- Response 1 ---
1. **“It’s raining cats and dogs.”**  
   **Latvian:** *Gāž kā ar spaiņiem.*  
   **Word-for-word back-translation:** “It’s pouring as if with buckets.”  
   **Use:** Said when it is raining very heavily.

2. **“Break a leg.”**  
   **Latvian:** *Ne pūkas, ne spalvas!*  
   **Word-for-word back-translation:** “Neither fluff nor feathers!”  
   **Use:** A traditional way to wish someone good luck, especially before a performance or competition.

3. **“Kill two birds with one stone.”**  
   **Latvian:** *Ar vienu šāvienu nošaut divus zaķus.*  
   **Word-for-word back-translation:** “To shoot two hares with one shot.”  
   **Use:** When one action achieves two goals.

4. **“Spill the beans.”**  
   **Latvian:** *Izpļāpāt noslēpumu.*  
   **Word-for-word back-translation:** “To blab a secret.”  
   **Use:** When someone reveals a secret, often accidentally or prematurely.

5. **“A blessing in disguise.”**  
   **Latvian:** *Katram mākonim ir zelta maliņa.*  
   **Word-for-word back-translation:** “Every cloud has a golden edge.”  
   **Use:** When something that initially seems bad eventually turns out to have a positive side.

Input tokens: 102 | Output tokens: 1196

### **My response** 

I chose Latvian since I'm native speaker. 
1. This was spot on. 
2. I have never heard that one, I think for actors you would just say "Salauz kāju" which is just direct word to word translation.
3. Correct idea but usually written other way around "Divi zaķi ar vienu šavienu/metienu".
4. This one isn't an idiom and I don't even know one I could fix it with. 
5. This one is odd and I don't even know how to change this.

In generally it did acceptable, especially since latvian language in general seems to be difficult for most AI models I have used.

#### 22. Generate a paragraph about a cat without using the letter 'e'/'E'.

Write a prompt that forces the AI to produce a three-sentence story about a cat while strictly forbidding it to use letter 'e'.

Use `gpt-5.4-mini` for this task.

Explain why your prompt is effective.

In [51]:
prompt = """Write a three-sentence story about a cat.
**Strict rule:** do not use the letter "e" or "E" anywhere, not in any word.
Before answering, check every word and replace any word containing "e" with a synonym without it.
Output only the three sentences."""

response, _ = chat(prompt)
story = response.choices[0].message.content
#print(story)
#print("Letters 'e' found:", story.lower().count("e"))

A sly cat sat on a warm mat by moonlight.  
It softly patted a tiny moth and purred with joy.  
By dawn, it darted off, a swift shadow in the yard.

Letters 'e' found: 4

### **My response**

Since model doesn't see letters and only tokens, as expected it didn't perform perfect in this task. As the additional print showed, it still let trough 4 "e". My prompt wasn't effective, I think most effective way would be to list out all the words with e and say to not use those.

#### 23. Extract JSON Data from the input text in two different ways.

Input Text: "Our trip to Tokyo starts on June 12th. We're staying for 5 nights at the Park Hyatt before heading to Kyoto."

Extract the information from the text above using two different methods:

1. Write a prompt asking the AI to return a JSON object with the fields `destination`, `start_date`, and `stay_duration`.

2. Instead of putting rules in the prompt, provide a specific JSON schema that defines the field names and their required data types `destination` (the first city), `start_date` (formatted as YYYY-MM-DD), and `stay_duration` (the number of nights as an integer).

Which method yields a more reliable result for a computer program to process, and why?

In [48]:
import json
trip = "Our trip to Tokyo starts on June 12th. We're staying for 5 nights at the Park Hyatt before heading to Kyoto."

prompt = f"Extract the information from this text and return ONLY a JSON object with the fields destination, start_date and stay_duration.\n\nText: {trip}"
response, _ = chat(prompt)
#print(response.choices[0].message.content)

{"destination":"Tokyo","start_date":"June 12th","stay_duration":"5 nights"}

In [49]:
schema = {
    "type": "object",
    "properties": {
        "destination": {"type": "string", "description": "The first city of the trip"},
        "start_date": {"type": "string", "description": "Start date in YYYY-MM-DD format"},
        "stay_duration": {"type": "integer", "description": "Number of nights"},
    },
    "required": ["destination", "start_date", "stay_duration"],
    "additionalProperties": False,
}

response = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": f"Extract the trip information from this text: {trip}"}],
    response_format={"type": "json_schema", "json_schema": {"name": "trip", "strict": True, "schema": schema}},
)
data = json.loads(response.choices[0].message.content)
#print(data, type(data["stay_duration"]))

{'destination': 'Tokyo', 'start_date': '2026-06-12', 'stay_duration': 5} <class 'int'>

### **My response**

JSON schema method is more reliable. Method 1 returned "June 12th" and "5 nights", as text. This might be better for human eyes but the programm would have to clean these up first. So returning "2026-06-12" and number 5 as an actual intiger is the best case since we dont want to change field names, types, etc. Only problem is that JSON guessed 2026 year, even thought we didn't give any year to enter. 

#### 24. Create a structured table comparing different programming languages.

Write a prompt that generates a table in this format (example below) with accurate information.

| Language    | Use Case               | Advantages                     | Disadvantages                |
|-------------|------------------------|---------------------------------|------------------------------|
| Python      | Data Science, AI       | Easy to learn, versatile       | Slower execution speed       |
| Java        | Enterprise Applications| Platform-independent, robust   | Verbose syntax               |
| JavaScript  | Web Development        | Fast, widely supported         | Limited for backend tasks    |

**Tip**: To enhance readability, you can copy and paste the generated table into a Markdown cell.

In [47]:
prompt = """Create a markdown table comparing 5 programming languages: Python, Java, JavaScript, C++ and Go.
Use exactly these columns: | Language | Use Case | Advantages | Disadvantages |
Keep each cell short (2-5 words) and make the information accurate.
Output only the table."""
response, _ = chat(prompt)
#print(response.choices[0].message.content)

| Language | Use Case | Advantages | Disadvantages |
|---|---|---|---|
| Python | Data science, scripting | Easy, huge ecosystem | Slower execution, GIL |
| Java | Enterprise applications, Android | Portable, strong tooling | Verbose, memory heavy |
| JavaScript | Web frontends, servers | Ubiquitous, flexible | Async complexity, weak typing |
| C++ | Systems, game engines | Fast, fine control | Complex, unsafe memory |
| Go | Cloud services, tooling | Simple, concurrent, fast | Limited generics, young ecosystem |

#### 25. Generate a quiz about machine learning basics.

Look at the example below and write a prompt that generates a quiz with identical formatting. Ensure the title formatting, list markers (a, b, c, d), and the correct answer indicator (*) are accurately replicated.

**Machine learning basics quiz**

1. Which of the following best describes supervised learning?  
a) The model learns from unlabeled data to find hidden patterns  
b) The model learns from labeled data to make predictions *  
c) The model is programmed with explicit rules  
d) The model randomly guesses outputs without training

2. Which of these is a common application of machine learning in everyday life?  
a) Sorting files manually on a computer  
b) Email spam filtering *  
c) Writing code by hand  
d) Using a calculator for arithmetic

3. What is overfitting in machine learning?  
a) When the model performs well on new, unseen data  
b) When the model is too simple and underperforms  
c) When the model learns the training data too well, including noise, reducing generalization *  
d) When the model runs faster than expected

4. Which ethical concern is commonly associated with machine learning algorithms?  
a) Increased hardware costs  
b) Bias in training data leading to unfair predictions *  
c) The inability to process large datasets  
d) Lack of available programming languages

5. Which of the following is NOT a type of machine learning?  
a) Supervised learning  
b) Reinforcement learning  
c) Unsupervised learning  
d) Manual learning *

In [70]:
example = """**Machine learning basics quiz**

1. Which of the following best describes supervised learning?  
a) The model learns from unlabeled data to find hidden patterns  
b) The model learns from labeled data to make predictions *  
c) The model is programmed with explicit rules  
d) The model randomly guesses outputs without training

2. Which of these is a common application of machine learning in everyday life?  
a) Sorting files manually on a computer  
b) Email spam filtering *  
c) Writing code by hand  
d) Using a calculator for arithmetic

3. What is overfitting in machine learning?  
a) When the model performs well on new, unseen data  
b) When the model is too simple and underperforms  
c) When the model learns the training data too well, including noise, reducing generalization *  
d) When the model runs faster than expected

4. Which ethical concern is commonly associated with machine learning algorithms?  
a) Increased hardware costs  
b) Bias in training data leading to unfair predictions *  
c) The inability to process large datasets  
d) Lack of available programming languages

5. Which of the following is NOT a type of machine learning?  
a) Supervised learning  
b) Reinforcement learning  
c) Unsupervised learning  
d) Manual learning *"""

prompt = f"""Create a new 5-question quiz about machine learning basics.
Copy the EXACT formatting of the example below:
- the title in bold on its own line
- numbered questions
- answers marked a), b), c), d), each on its own line
- exactly one correct answer per question, marked with " *" at the end
Use different questions than the example.

Example:
{example}"""
response, _ = chat(prompt)
#print(response.choices[0].message.content)

**Machine learning basics quiz**

1. What is the main goal of machine learning?  
a) To replace all human decision-making  
b) To allow systems to learn patterns from data and improve at tasks *  
c) To eliminate the need for computers  
d) To store data without processing it

2. Which of the following is an example of unsupervised learning?  
a) Predicting house prices from labeled examples  
b) Grouping customers into clusters based on similar behavior *  
c) Training a model to recognize handwritten digits with labels  
d) Teaching a system using exact step-by-step rules

3. What is a training dataset used for?  
a) To evaluate the final model only  
b) To teach the model patterns from examples *  
c) To replace the model entirely  
d) To remove all errors from data automatically

4. What does a feature represent in machine learning?  
a) A measurable input used by the model *  
b) The final prediction the model makes  
c) The error rate of the model  
d) A type of algorithm for clustering

5. Why is a test set important?  
a) It is used to train the model faster  
b) It helps measure how well the model performs on new data *  
c) It guarantees the model will never overfit  
d) It changes the model’s internal rules

#### 26. Create a comprehensive online course outline for beginners.

Write a prompt that generates entire course outline, following the example provided below. Ensure the prompt is detailed enough to cover all sections, with clear formatting and practical guidance. 

Sections to include in course outline:
- Course overview - description, objectives, target audience.
- Module breakdown - 5 modules with title, short description, three key learning outcomes and an interactive assignment in the end of each module.
- Assessment plan - a table outlining the assessment methods with description and weighting.
- Certification criteria - requirements to earn a certificate.
- Marketing strategy - step-by-step guide for promoting the course.

Include a detailed description of the type of information each section should contain. Maintain the **same formatting**, **section titles**, and **course name**. While the generated descriptions, modules and marketing strategy steps may differ and incorporate new ideas and concepts, it must remain aligned with the theme of the task.

Put the final output of your API call in a Markdown cell. 

#### Example:
---
---
## Course Outline: Mastering Prompt Engineering for AI Applications

---

### Course Overview
This beginner-friendly course is designed to equip learners with the foundational skills and knowledge required to craft effective prompts for AI applications. By mastering prompt engineering, students will learn how to optimize AI models for various tasks, including text generation, data analysis, and creative problem-solving. The course is ideal for individuals interested in AI, developers, data scientists, and anyone looking to leverage AI tools in their personal or professional projects.

#### Objectives
- Understand the principles of prompt engineering and its role in AI workflows.
- Learn techniques to design, test, and refine prompts for optimal AI performance.
- Apply prompt engineering skills to real-world scenarios and projects.

#### Target Audience
- Beginners with no prior experience in AI or programming.
- Professionals seeking to integrate AI into their workflows.
- Students and enthusiasts curious about AI capabilities.

---

### Module Breakdown

#### Module 1: Introduction to Prompt Engineering
- **Description:** This module introduces the concept of prompt engineering, its importance in AI applications, and the basics of interacting with AI models.
- **Key Learning Outcomes:**
  1. Define prompt engineering and its significance in AI workflows.
  2. Understand the structure and components of effective prompts.
  3. Explore real-world use cases of prompt engineering.
- **Interactive Assignment:** Write a simple prompt for a text-generation AI model to generate a short story.

---

#### Module 2: Principles of Effective Prompt Design
- **Description:** Learn the core principles and techniques for designing prompts that yield accurate and relevant AI outputs.
- **Key Learning Outcomes:**
  1. Identify factors that influence AI responses (e.g., context, specificity, tone).
  2. Explore strategies for iterative prompt refinement.
  3. Understand common pitfalls and how to avoid them.
- **Interactive Assignment:** Design and test three variations of a prompt for a chatbot to answer customer queries effectively.

---

#### Module 3: Advanced Techniques in Prompt Engineering
- **Description:** Dive deeper into advanced methods, including chaining prompts, dynamic prompts, and leveraging AI tools for complex tasks.
- **Key Learning Outcomes:**
  1. Learn how to chain prompts for multi-step tasks.
  2. Explore dynamic prompt techniques for adaptive AI responses.
  3. Understand how to integrate external data sources into prompts.
- **Interactive Assignment:** Create a multi-step prompt to generate a detailed report based on user-provided data.

---

#### Module 4: Applications of Prompt Engineering Across Industries
- **Description:** Discover how prompt engineering is applied in various industries, such as healthcare, education, marketing, and software development.
- **Key Learning Outcomes:**
  1. Analyze industry-specific use cases of prompt engineering.
  2. Learn how to tailor prompts for different domains and objectives.
  3. Explore ethical considerations in prompt engineering.
- **Interactive Assignment:** Develop a prompt for a marketing campaign, including product descriptions and social media posts.

---

#### Module 5: Building and Testing AI Projects with Prompt Engineering
- **Description:** Apply your skills to design, test, and deploy AI projects using prompt engineering techniques.
- **Key Learning Outcomes:**
  1. Plan and execute an AI project using prompt engineering.
  2. Test and refine prompts for optimal project outcomes.
  3. Document and present your project effectively.
- **Interactive Assignment:** Create a mini-project using prompt engineering to solve a real-world problem (e.g., automating customer support).

---

### Assessment Plan

| Assessment Method      | Description                          | Weightage (%) |
|------------------------|--------------------------------------|---------------|
| Quizzes               | Multiple-choice quizzes after each module | 30%          |
| Assignments           | Practical exercises for each module  | 40%           |
| Final Project         | A comprehensive project combining all skills | 20%          |
| Participation         | Active participation in discussions  | 10%           |

---

### Certification Criteria
To earn a certificate of completion, students must:
1. Achieve a minimum score of 70% across all assessments.
2. Submit all interactive assignments and the final project.
3. Actively participate in at least 80% of discussion forums and live sessions.

---

### Marketing Strategy

#### Step 1: Social Media Campaigns
- Create engaging posts highlighting the course benefits, modules, and outcomes.
- Share testimonials and success stories from beta testers or early participants.
- Use targeted ads on platforms like LinkedIn, Facebook, and Instagram.

#### Step 2: Email Marketing
- Build an email list of potential learners (e.g., professionals, students).
- Send a series of emails introducing the course, its benefits, and enrollment deadlines.
- Include exclusive early-bird discounts and referral incentives.

#### Step 3: Partnerships
- Collaborate with universities, coding bootcamps, and AI communities to promote the course.
- Partner with influencers and thought leaders in the AI space for endorsements.

#### Step 4: Content Creation
- Publish blog posts, videos, and webinars on topics related to prompt engineering.
- Share free resources (e.g., mini-guides, templates) to attract interest.
- Host live Q&A sessions to engage with the audience and answer questions about the course.

#### Step 5: Testimonials and Reviews
- Collect feedback from early participants and showcase their success stories.
- Encourage learners to share their experiences on social media and review platforms.

---

In [74]:
example = """
---
## Course Outline: Mastering Prompt Engineering for AI Applications

---

### Course Overview
This beginner-friendly course is designed to equip learners with the foundational skills and knowledge required to craft effective prompts for AI applications. By mastering prompt engineering, students will learn how to optimize AI models for various tasks, including text generation, data analysis, and creative problem-solving. The course is ideal for individuals interested in AI, developers, data scientists, and anyone looking to leverage AI tools in their personal or professional projects.

#### Objectives
- Understand the principles of prompt engineering and its role in AI workflows.
- Learn techniques to design, test, and refine prompts for optimal AI performance.
- Apply prompt engineering skills to real-world scenarios and projects.

#### Target Audience
- Beginners with no prior experience in AI or programming.
- Professionals seeking to integrate AI into their workflows.
- Students and enthusiasts curious about AI capabilities.

---

### Module Breakdown

#### Module 1: Introduction to Prompt Engineering
- **Description:** This module introduces the concept of prompt engineering, its importance in AI applications, and the basics of interacting with AI models.
- **Key Learning Outcomes:**
  1. Define prompt engineering and its significance in AI workflows.
  2. Understand the structure and components of effective prompts.
  3. Explore real-world use cases of prompt engineering.
- **Interactive Assignment:** Write a simple prompt for a text-generation AI model to generate a short story.

---

#### Module 2: Principles of Effective Prompt Design
- **Description:** Learn the core principles and techniques for designing prompts that yield accurate and relevant AI outputs.
- **Key Learning Outcomes:**
  1. Identify factors that influence AI responses (e.g., context, specificity, tone).
  2. Explore strategies for iterative prompt refinement.
  3. Understand common pitfalls and how to avoid them.
- **Interactive Assignment:** Design and test three variations of a prompt for a chatbot to answer customer queries effectively.

---

#### Module 3: Advanced Techniques in Prompt Engineering
- **Description:** Dive deeper into advanced methods, including chaining prompts, dynamic prompts, and leveraging AI tools for complex tasks.
- **Key Learning Outcomes:**
  1. Learn how to chain prompts for multi-step tasks.
  2. Explore dynamic prompt techniques for adaptive AI responses.
  3. Understand how to integrate external data sources into prompts.
- **Interactive Assignment:** Create a multi-step prompt to generate a detailed report based on user-provided data.

---

#### Module 4: Applications of Prompt Engineering Across Industries
- **Description:** Discover how prompt engineering is applied in various industries, such as healthcare, education, marketing, and software development.
- **Key Learning Outcomes:**
  1. Analyze industry-specific use cases of prompt engineering.
  2. Learn how to tailor prompts for different domains and objectives.
  3. Explore ethical considerations in prompt engineering.
- **Interactive Assignment:** Develop a prompt for a marketing campaign, including product descriptions and social media posts.

---

#### Module 5: Building and Testing AI Projects with Prompt Engineering
- **Description:** Apply your skills to design, test, and deploy AI projects using prompt engineering techniques.
- **Key Learning Outcomes:**
  1. Plan and execute an AI project using prompt engineering.
  2. Test and refine prompts for optimal project outcomes.
  3. Document and present your project effectively.
- **Interactive Assignment:** Create a mini-project using prompt engineering to solve a real-world problem (e.g., automating customer support).

---

### Assessment Plan

| Assessment Method      | Description                          | Weightage (%) |
|------------------------|--------------------------------------|---------------|
| Quizzes               | Multiple-choice quizzes after each module | 30%          |
| Assignments           | Practical exercises for each module  | 40%           |
| Final Project         | A comprehensive project combining all skills | 20%          |
| Participation         | Active participation in discussions  | 10%           |

---

### Certification Criteria
To earn a certificate of completion, students must:
1. Achieve a minimum score of 70% across all assessments.
2. Submit all interactive assignments and the final project.
3. Actively participate in at least 80% of discussion forums and live sessions.

---

### Marketing Strategy

#### Step 1: Social Media Campaigns
- Create engaging posts highlighting the course benefits, modules, and outcomes.
- Share testimonials and success stories from beta testers or early participants.
- Use targeted ads on platforms like LinkedIn, Facebook, and Instagram.

#### Step 2: Email Marketing
- Build an email list of potential learners (e.g., professionals, students).
- Send a series of emails introducing the course, its benefits, and enrollment deadlines.
- Include exclusive early-bird discounts and referral incentives.

#### Step 3: Partnerships
- Collaborate with universities, coding bootcamps, and AI communities to promote the course.
- Partner with influencers and thought leaders in the AI space for endorsements.

#### Step 4: Content Creation
- Publish blog posts, videos, and webinars on topics related to prompt engineering.
- Share free resources (e.g., mini-guides, templates) to attract interest.
- Host live Q&A sessions to engage with the audience and answer questions about the course.

#### Step 5: Testimonials and Reviews
- Collect feedback from early participants and showcase their success stories.
- Encourage learners to share their experiences on social media and review platforms.

---"""

prompt = f"""Create a complete online course outline for beginners called "Mastering Prompt Engineering for AI Applications".

Use **exactly the same formatting, section titles and course name** as the example below. Include:
1. **Course Overview**: a description paragraph, then "Objectives" (3 bullets) and "Target Audience" (3 bullets).
2. **Module Breakdown**: 5 modules, each with a title, a short **Description**, three numbered **Key Learning Outcomes** and an **Interactive Assignment**. Separate modules with ---.
3. **Assessment Plan**: a markdown table with the columns Assessment Method, Description and Weightage (%), adding up to 100%.
4. **Certification Criteria**: a numbered list of requirements.
5. **Marketing Strategy**: 5 steps, each with a heading and 2-3 bullet points.

The module topics, descriptions and marketing steps should be NEW ideas, not copied from the example, but must stay about prompt engineering.

Example:
{example}"""
response, _ = chat(prompt, model="gpt-5.6-luna")
#print(response.choices[0].message.content)


---
## Course Outline: Mastering Prompt Engineering for AI Applications

---

### Course Overview
This beginner-friendly course provides a practical foundation in prompt engineering for learners who want to use generative AI effectively, responsibly, and consistently. Students will learn how to communicate task requirements clearly, provide useful context, control output formats, evaluate AI responses, and improve prompts through structured experimentation. The course combines guided demonstrations, hands-on exercises, and a final application project so learners can transfer prompt engineering techniques to writing, research, productivity, business, and creative workflows.

#### Objectives
- Understand how AI language models interpret instructions, context, examples, and constraints.
- Develop, test, and improve prompts for reliable and purpose-driven outputs.
- Apply prompt engineering methods to practical AI applications while considering accuracy, privacy, bias, and responsible use.

#### Target Audience
- Beginners who want to use generative AI tools without requiring programming experience.
- Professionals, educators, entrepreneurs, and students seeking to improve AI-assisted workflows.
- Creators and technology enthusiasts interested in building practical AI-powered solutions.

---

### Module Breakdown

#### Module 1: How AI Models Interpret Prompts
- **Description:** This module explains the basic interaction between users and generative AI systems. Learners will examine how instructions, context, intent, and wording influence model responses, while developing realistic expectations about model capabilities and limitations.
- **Key Learning Outcomes:**
  1. Describe the relationship between user instructions, context, and AI-generated outputs.
  2. Distinguish between vague, underspecified, and well-scoped prompts.
  3. Identify situations where AI responses require verification or human judgment.
- **Interactive Assignment:** Compare three prompts for the same task, record the differences in the AI outputs, and write a short analysis explaining which prompt produced the most useful result and why.

---

#### Module 2: Designing Prompts with Roles, Context, and Constraints
- **Description:** Learners will explore a reusable framework for constructing prompts with a clear objective, relevant background information, desired audience, tone, boundaries, and output requirements. The module focuses on making prompts precise without adding unnecessary complexity.
- **Key Learning Outcomes:**
  1. Construct prompts that clearly define goals, audience, inputs, and expected results.
  2. Use roles, examples, constraints, and formatting instructions to guide AI behavior.
  3. Create reusable prompt templates for recurring personal or professional tasks.
- **Interactive Assignment:** Build a prompt template that transforms unstructured notes into a well-organized deliverable, such as a meeting summary, lesson plan, project brief, or product description.

---

#### Module 3: Prompt Testing, Evaluation, and Iteration
- **Description:** This module introduces a systematic process for improving prompts through experimentation. Learners will establish evaluation criteria, compare outputs, identify failure patterns, and revise prompts using evidence rather than guesswork.
- **Key Learning Outcomes:**
  1. Define quality criteria for evaluating accuracy, relevance, completeness, tone, and format.
  2. Use controlled prompt variations to determine which instructions improve results.
  3. Diagnose common output problems, including omissions, unsupported claims, repetition, and inconsistent formatting.
- **Interactive Assignment:** Conduct a small prompt experiment using at least four prompt versions, score the outputs against a self-created rubric, and submit a revised prompt with an explanation of the changes.

---

#### Module 4: Prompting for Multi-Step Workflows and Structured Outputs
- **Description:** Learners will move beyond single-turn requests and design prompts for complex workflows. Topics include breaking large tasks into stages, requesting structured responses, using examples, and asking AI to clarify missing information before completing a task.
- **Key Learning Outcomes:**
  1. Decompose complex objectives into manageable prompt stages.
  2. Request structured outputs such as tables, checklists, classifications, schemas, or action plans.
  3. Design prompts that support review, refinement, and human approval within an AI workflow.
- **Interactive Assignment:** Create a multi-step workflow that takes a realistic input, processes it through at least three stages, and produces a structured final output suitable for a chosen application.

---

#### Module 5: Responsible Prompt Engineering in Real-World Applications
- **Description:** This module focuses on applying prompt engineering responsibly across workplace, educational, creative, and customer-facing scenarios. Learners will consider privacy, sensitive information, bias, transparency, intellectual property, and the need for human oversight when designing AI-assisted processes.
- **Key Learning Outcomes:**
  1. Apply prompt engineering techniques to a practical use case with clearly defined users and outcomes.
  2. Recognize privacy, fairness, security, and reliability risks in AI-generated results.
  3. Document prompts, assumptions, evaluation results, and safeguards for responsible use.
- **Interactive Assignment:** Develop and present a mini AI application concept, including its purpose, prompt workflow, sample inputs and outputs, evaluation checklist, risk analysis, and proposed human review process.

---

### Assessment Plan

| Assessment Method      | Description                                                    | Weightage (%) |
|------------------------|----------------------------------------------------------------|---------------|
| Knowledge Checks       | Short quizzes measuring understanding of concepts and terminology after each module | 20% |
| Prompt Design Exercises | Practical activities focused on writing, formatting, and adapting prompts | 25% |
| Prompt Evaluation Report | A documented experiment comparing prompt versions against defined quality criteria | 20% |
| Workflow Assignment    | A multi-step prompt workflow that produces a structured and reviewable result | 15% |
| Final Application Project | A responsible prompt engineering solution addressing a realistic user or organizational need | 15% |
| Participation          | Contributions to peer reviews, discussions, and feedback activities | 5% |
| **Total**              |                                                                | **100%** |

---

### Certification Criteria
To earn a certificate of completion, students must:
1. Achieve an overall assessment score of at least 70%.
2. Complete all five module assignments, including the prompt evaluation report and workflow assignment.
3. Submit a final application project that includes prompts, sample outputs, evaluation criteria, and responsible-use considerations.
4. Participate in at least 80% of required peer-review, discussion, or live learning activities.
5. Demonstrate the ability to revise a prompt based on output quality and documented evaluation results.

---

### Marketing Strategy

#### Step 1: Position the Course Around Practical Outcomes
- Promote the course as a beginner-friendly way to make AI outputs clearer, more consistent, and more useful.
- Highlight practical outcomes such as reusable prompt templates, structured workflows, and an application project.
- Use audience-specific messaging for professionals, educators, students, creators, and small business owners.

#### Step 2: Offer an Interactive Lead Magnet
- Create a free prompt improvement checklist, beginner template pack, or prompt evaluation rubric.
- Require a simple email signup to download the resource and introduce interested learners to the course.
- Include a short self-assessment that helps visitors identify their current prompt engineering skill level.

#### Step 3: Demonstrate Before-and-After Results
- Publish short videos showing how a vague instruction becomes a precise, high-quality prompt.
- Share side-by-side examples of outputs produced with different prompt structures and constraints.
- Use anonymized learner projects to demonstrate how prompt engineering supports real tasks.

#### Step 4: Build Learning Community Partnerships
- Collaborate with productivity groups, educator networks, career communities, technology newsletters, and professional associations.
- Provide partner organizations with a private workshop, referral code, or co-branded introductory session.
- Invite community leaders to discuss how they use AI tools and connect their experiences to course topics.

#### Step 5: Create a Trust-Focused Enrollment Campaign
- Emphasize responsible AI use, verification practices, privacy awareness, and human oversight.
- Host a live introductory class where prospective learners can submit prompts for improvement.
- Use learner feedback, project showcases, completion statistics, and transparent course requirements to support enrollment decisions.

---
```

####